In [0]:
### Célula 6 - gravar em `squad1.ecommerce_enderecos`

jdbc_url = (
    f"jdbc:sqlserver://{sql_host}:1433;"
    f"database={sql_database};encrypt=true;trustServerCertificate=false;loginTimeout=30;"
)
jdbc_properties = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

(
    df.write.format("sqlserver")
    .option("host", sql_host)
    .option("port", "1433")
    .option("database", sql_database)
    .option("dbtable", "squad1.ecommerce_enderecos")
    .option("user", sql_username)
    .option("password", sql_password)
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .mode("overwrite")
    .save()
)

In [0]:
### Célula 7 - conferir a gravação

spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT COUNT(*) AS total FROM squad1.ecommerce_enderecos) t",
    properties=jdbc_properties,
).show()

In [0]:
## Passo 8 - Detectar novos arquivos (tempo real)
import time

def check_real_time_data(max_attempts=5, wait_seconds=30):
    path = f"{base_path}/real-time-data"
    for attempt in range(1, max_attempts + 1):
        files_df = (
            spark.read.format("binaryFile")
            .options(**adls_opts)
            .option("recursiveFileLookup", "true")
            .load(path)
            .select("path", "length", "modificationTime")
        )
        total = files_df.count()
        if total > 0:
            print(f"Tentativa {attempt}: {total} arquivo(s) encontrado(s).")
            return files_df
        print(f"Tentativa {attempt}/{max_attempts}: nada em {path} ainda.")
        if attempt < max_attempts:
            time.sleep(wait_seconds)
    print("Nada encontrado ainda. Rode de novo mais tarde.")
    return None

arquivos = check_real_time_data()
if arquivos is not None:
    display(arquivos)